In [1]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    x = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(x, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return x, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

In [2]:
# 调用框架中现有的API来读取数据
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)


batch_size = 10
data_iter = load_array((features, labels), batch_size)

next(iter(data_iter))

[tensor([[ 0.5265, -0.2765],
         [-0.2160, -0.3579],
         [ 0.0332,  0.4362],
         [-0.0705, -0.1646],
         [ 1.4283,  0.7548],
         [-0.3389,  0.1634],
         [ 0.3000,  0.8394],
         [-0.5037, -1.1197],
         [ 0.0138,  1.6076],
         [-0.1831,  0.1797]]),
 tensor([[ 6.1824],
         [ 4.9759],
         [ 2.7640],
         [ 4.6144],
         [ 4.5000],
         [ 2.9602],
         [ 1.9574],
         [ 6.9939],
         [-1.2372],
         [ 3.2050]])]

In [4]:
# 使用框架的预定义好的层
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

In [5]:
# 初始化模型参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [6]:
# 均方误差使用的是MSELoss类，平方范数
loss = nn.MSELoss()

In [7]:
# 实例化 SGD 实例
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [8]:
# 训练过程
num_epochs = 3

for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)  # 自带模型参数
        trainer.zero_grad()  # 优化器清0梯度
        l.backward()         # 计算梯度
        trainer.step()       # 模型更新
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l : f}')

epoch 1, loss  0.000200
epoch 2, loss  0.000097
epoch 3, loss  0.000097
